# Lesson 42: FastAPI

**Data Science with Python**

This notebook covers building ML APIs with FastAPI.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/data-science-notebooks/blob/main/L42_FastAPI/colab/L42_FastAPI_Notebook.ipynb)

## 1. Setup and Imports

In [ ]:
# Install FastAPI and uvicorn (run once)
# !pip install fastapi uvicorn pydantic

from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression

print("Libraries loaded successfully!")

## 2. Basic FastAPI App Structure

In [ ]:
# Basic FastAPI structure (for reference - run in separate file)
api_code = '''
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def read_root():
    return {"message": "Hello, World!"}

@app.get("/health")
def health_check():
    return {"status": "healthy"}
'''

print("Basic FastAPI app structure:")
print(api_code)

## 3. Pydantic Models for Validation

In [ ]:
# Define request/response models
class PredictionRequest(BaseModel):
    feature1: float
    feature2: float
    feature3: float

class PredictionResponse(BaseModel):
    prediction: int
    probability: float

# Test the models
request = PredictionRequest(feature1=1.5, feature2=2.0, feature3=0.5)
print(f"Request model: {request}")
print(f"As dict: {request.dict()}")

## 4. Training a Sample Model

In [ ]:
# Train a simple model
np.random.seed(42)
X = np.random.randn(100, 3)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

model = LogisticRegression()
model.fit(X, y)

# Save model
joblib.dump(model, 'sample_model.pkl')
print("Model trained and saved!")
print(f"Accuracy: {model.score(X, y):.2%}")

## 5. Complete ML API Structure

In [ ]:
# Complete ML API code (save as main.py)
ml_api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

# Initialize app
app = FastAPI(title="ML Prediction API")

# Load model at startup
model = joblib.load("sample_model.pkl")

# Define schemas
class PredictionRequest(BaseModel):
    feature1: float
    feature2: float
    feature3: float

class PredictionResponse(BaseModel):
    prediction: int
    probability: float

@app.get("/")
def root():
    return {"message": "ML Prediction API"}

@app.post("/predict", response_model=PredictionResponse)
def predict(request: PredictionRequest):
    # Convert to array
    features = np.array([[request.feature1, request.feature2, request.feature3]])
    
    # Make prediction
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0].max()
    
    return PredictionResponse(
        prediction=int(prediction),
        probability=float(probability)
    )
'''

print("Complete ML API structure:")
print(ml_api_code)

## 6. Testing the API

In [ ]:
# Simulate API call locally
model = joblib.load('sample_model.pkl')

def simulate_predict(feature1, feature2, feature3):
    """Simulate the /predict endpoint"""
    features = np.array([[feature1, feature2, feature3]])
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0].max()
    return {
        "prediction": int(prediction),
        "probability": float(probability)
    }

# Test predictions
test_cases = [
    (1.0, 1.0, 0.5),
    (-1.0, -1.0, 0.0),
    (0.5, -0.5, 0.0)
]

print("Simulated API Responses:")
for features in test_cases:
    result = simulate_predict(*features)
    print(f"  Input: {features} -> {result}")

## 7. Exercise: Add Input Validation

Add validation to ensure features are within valid ranges.

<details>
<summary>Click for Solution</summary>

```python
from pydantic import BaseModel, validator, Field

class ValidatedRequest(BaseModel):
    feature1: float = Field(..., ge=-10, le=10)
    feature2: float = Field(..., ge=-10, le=10)
    feature3: float = Field(..., ge=-10, le=10)
    
    @validator('feature1', 'feature2', 'feature3')
    def check_not_zero(cls, v):
        if abs(v) < 0.001:
            raise ValueError('Features cannot be zero')
        return v

# Test validation
try:
    valid = ValidatedRequest(feature1=5.0, feature2=3.0, feature3=1.0)
    print(f"Valid: {valid}")
except Exception as e:
    print(f"Error: {e}")

try:
    invalid = ValidatedRequest(feature1=100.0, feature2=3.0, feature3=1.0)
except Exception as e:
    print(f"Caught validation error: {e}")
```
</details>

In [ ]:
# Your solution here


## Key Takeaways

1. **FastAPI** provides modern, fast API development
2. **Pydantic** models handle validation automatically
3. Load models **once** at startup for efficiency
4. Use **/docs** for automatic API documentation
5. Run with: `uvicorn main:app --reload`